######## Whole pipeline is in this file ##########

## EDA ##

In [1]:
import duckdb
import json
import pandas as pd
import streamlit as st
import requests
import os
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.getenv("API_KEY")
API_KEY is not None

True

In [2]:
#loading raw file
df=pd.read_csv('data/revenues_per_day_fm.csv')



In [3]:
#file size check
df.shape

(337818, 6)

In [4]:
#first 5 rows
df.head()

,id,date,title,revenue,theaters,distributor
0,8b19ad43-3a7e-b14b-49e9-1f7a0eb1568e,2004-09-20,Sky Captain and the World of Tomorrow,925482,3170.0,Paramount Pictures
1,481fc700-fcdd-1919-c53c-09fcd423a596,2004-09-20,Resident Evil: Apocalypse,643680,3284.0,Screen Gems
2,06719cc2-c05e-8c0b-56b6-b8b13e056509,2004-09-20,Mr 3000,425375,2736.0,Walt Disney Studios Motion Pictures
3,3be8e4a8-2716-00be-80be-eb342ba3cbe7,2004-09-20,Wimbledon,416970,2034.0,Universal Pictures
4,c5f26b59-6a64-ae8e-8f58-7cefb09eadfe,2004-09-20,Cellular,412000,2749.0,New Line Cinema


In [5]:
df.describe()

,revenue,theaters
count,3.378180e+05,337657.000000
mean,6.071642e+05,848.611597
std,2.323203e+06,1137.442240
min,0.000000e+00,1.000000
25%,6.592000e+03,45.000000
50%,4.077500e+04,226.000000
75%,3.029205e+05,1402.000000
max,1.574616e+08,4931.000000


In [6]:
df.dtypes

id                 str
date               str
title              str
revenue          int64
theaters       float64
distributor        str
dtype: object

In [7]:
#how many nulls by columns
df.isna().sum()

#0.05% nulls in theaters column, and 0.07% of nulls in distributor column.
#edit: after correctly labeling nulls in distributor columns it's 2,3%

id               0
date             0
title            0
revenue          0
theaters       161
distributor    225
dtype: int64

In [8]:
#fully duplicated rows
df.duplicated().sum()

np.int64(0)

In [9]:
#errors in data type change
df["date"].isna().sum()    


np.int64(0)

In [10]:
#connecting duckdb and creating table for sql
con = duckdb.connect('data/movies.duckdb')
con.sql("CREATE OR REPLACE TABLE raw_revenues AS SELECT * FROM read_csv_auto('data/revenues_per_day_fm.csv')")

In [11]:
con.sql("DESCRIBE raw_revenues")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ title       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ revenue     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ theaters    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ distributor │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [12]:
#how many rows (should be the same as in pandas)
con.sql("SELECT COUNT(*) FROM raw_revenues")


┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       337818 │
└──────────────┘

In [13]:
con.sql("SELECT MIN(date), MAX(date) FROM raw_revenues")

┌────────────┬────────────┐
│ min(date)  │ max(date)  │
│    date    │    date    │
├────────────┼────────────┤
│ 2000-01-01 │ 2023-03-06 │
└────────────┴────────────┘

In [14]:
#how many unique titles
con.sql("SELECT COUNT(DISTINCT(title)) FROM raw_revenues")

┌───────────────────────┐
│ count(DISTINCT title) │
│         int64         │
├───────────────────────┤
│                  6545 │
└───────────────────────┘

In [15]:
con.sql("SELECT COUNT(id) FROM raw_revenues GROUP BY id HAVING COUNT(id)>1")

┌───────────┐
│ count(id) │
│   int64   │
└───────────┘
   0 rows  

In [16]:
con.sql("SELECT title, date, COUNT(*) FROM raw_revenues GROUP BY title, date HAVING COUNT(*)>1")

┌─────────┬──────┬──────────────┐
│  title  │ date │ count_star() │
│ varchar │ date │    int64     │
└─────────┴──────┴──────────────┘
             0 rows            

In [17]:
con.sql("SELECT id, title, revenue FROM raw_revenues WHERE revenue <= 0 ORDER BY revenue")

┌──────────────────────────────────────┬───────────────────────────────────────────┬─────────┐
│                  id                  │                   title                   │ revenue │
│               varchar                │                  varchar                  │  int64  │
├──────────────────────────────────────┼───────────────────────────────────────────┼─────────┤
│ 4cd24fa3-e62d-ded1-cd24-035a3afa4b01 │ White Snake                               │       0 │
│ b36b7203-7ed6-9673-0be7-a272b9343c62 │ Le Cercle Rouge2021 Re-release            │       0 │
│ b54493e6-4d31-1f1b-45ea-e7855d9e3c60 │ The Times of Bill Cunningham              │       0 │
│ e8396d5b-7ae8-5559-ae54-c8445c696014 │ Incitement                                │       0 │
│ 08af2a23-92ea-a3e0-0a26-4b759cc301c4 │ Counter Column                            │       0 │
│ 89bc5398-9498-e752-e058-bbb0fa09bcdd │ The Gentlemen                             │       0 │
│ e2f0dba7-f4d8-0ee4-926b-f0595d6d6a43 │ Incitemen

In [18]:
con.sql("SELECT id, date, title, revenue FROM raw_revenues ORDER BY revenue DESC LIMIT 5")
#Avengers: Endgame earned 157mln$ which is true, so revenue shows in usd.
#This also means that there's 63 rows of movies earning 0 usd in the record.

┌──────────────────────────────────────┬────────────┬────────────────────────────────────────────┬───────────┐
│                  id                  │    date    │                   title                    │  revenue  │
│               varchar                │    date    │                  varchar                   │   int64   │
├──────────────────────────────────────┼────────────┼────────────────────────────────────────────┼───────────┤
│ c5477391-de7d-0cb3-4738-bb9e971d6b53 │ 2019-04-26 │ Avengers: Endgame                          │ 157461641 │
│ e5f9c957-c088-7070-b32c-4e949c0c7d3d │ 2021-12-17 │ Spider-Man: No Way Home                    │ 121850000 │
│ 03ec9798-2fb1-b3e9-b711-3f0a6932f82f │ 2015-12-18 │ Star Wars: Episode VII - The Force Awakens │ 119119282 │
│ c88d93b6-2571-0f47-6e89-e40b0835361a │ 2019-04-27 │ Avengers: Endgame                          │ 109264122 │
│ b9a734bf-fe3f-42cc-404e-8cade0cfc075 │ 2018-04-27 │ Avengers: Infinity War                     │ 106334939 │
└

In [19]:
con.sql("SELECT COUNT(DISTINCT distributor) AS cdis, title FROM raw_revenues GROUP BY title HAVING cdis>1")

┌───────┬────────────────────────────────────────────────────────────────────┐
│ cdis  │                               title                                │
│ int64 │                              varchar                               │
├───────┼────────────────────────────────────────────────────────────────────┤
│     2 │ Amazing Grace                                                      │
│     2 │ No Escape                                                          │
│     2 │ Walking with Herb                                                  │
│     2 │ Mission Stories                                                    │
│     2 │ Jujutsu Kaisen 0: The Movie                                        │
│     2 │ The Kid                                                            │
│     2 │ Serenity                                                           │
│     2 │ Infamous                                                           │
│     2 │ The Lost City                             

In [20]:
con.sql("SELECT distributor, COUNT(*) FROM raw_revenues GROUP BY distributor ORDER BY COUNT(*) ")

┌─────────────────────────────────────┬──────────────┐
│             distributor             │ count_star() │
│               varchar               │    int64     │
├─────────────────────────────────────┼──────────────┤
│ CFI Releasing                       │            1 │
│ FilmDemic                           │            1 │
│ Amplify                             │            1 │
│ Utopia                              │            1 │
│ Universal Home Video                │            1 │
│ PBS Distribution                    │            1 │
│ International Film Circuit          │            1 │
│ Big World Pictures                  │            1 │
│ Shorts International                │            1 │
│ Kicking + Screening Media Group     │            1 │
│           ·                         │            · │
│           ·                         │            · │
│           ·                         │            · │
│ The Weinstein Company               │        10120 │
│ Roadside

In [21]:
con.sql("SELECT title, MIN(date), MAX(date), MAX(date)-MIN(date) AS diff FROM raw_revenues GROUP BY title ORDER BY diff DESC")

┌────────────────────────────────────────────────────────┬────────────┬────────────┬───────┐
│                         title                          │ min(date)  │ max(date)  │ diff  │
│                        varchar                         │    date    │    date    │ int64 │
├────────────────────────────────────────────────────────┼────────────┼────────────┼───────┤
│ Charlie's Angels                                       │ 2000-11-03 │ 2020-01-16 │  7013 │
│ The Kid                                                │ 2000-07-07 │ 2019-04-18 │  6859 │
│ Pinocchio                                              │ 2002-12-25 │ 2021-03-04 │  6644 │
│ The Lost City                                          │ 2006-04-28 │ 2022-06-23 │  5900 │
│ The Grudge                                             │ 2004-10-22 │ 2020-02-27 │  5606 │
│ A Grin Without A Cat                                   │ 2002-05-01 │ 2017-07-19 │  5558 │
│ Hellboy                                                │ 2004-04-02 

In [22]:
con.sql("SELECT id, title, date FROM raw_revenues WHERE title IN ('Charlie''s Angels', 'The Kid', 'Pinocchio')")
#id is unique for title and day, not just for distinct movies.

┌──────────────────────────────────────┬──────────────────┬────────────┐
│                  id                  │      title       │    date    │
│               varchar                │     varchar      │    date    │
├──────────────────────────────────────┼──────────────────┼────────────┤
│ f404b371-90b2-826b-6514-e6312d0a9f8d │ Charlie's Angels │ 2000-12-29 │
│ 3a733fdb-c80f-55e1-c90e-09a6c9e0ec07 │ The Kid          │ 2000-08-08 │
│ b57cfc95-88de-90fb-b972-f078ab08de22 │ The Kid          │ 2019-03-10 │
│ 936b83fb-eef3-b64a-b041-3a993162d5f0 │ Charlie's Angels │ 2019-11-23 │
│ d189d53c-6597-94f1-200c-0d723e9a234f │ Charlie's Angels │ 2020-01-13 │
│ e8663642-f860-0a7a-f1eb-0e6bf3b617ac │ The Kid          │ 2000-07-30 │
│ 49f9715a-19a4-c684-146e-493dd42fc22e │ Charlie's Angels │ 2000-12-27 │
│ d582dc1c-e0e4-27c2-e03c-43be3f5d96bc │ Charlie's Angels │ 2019-12-27 │
│ 536fc148-d9ee-7d45-0113-f00fb8be4e5c │ The Kid          │ 2000-08-29 │
│ d12c840e-0220-06dd-3bf0-de88370070cc │ Charlie's 

In [23]:
#some movies are remakes, and some are re-releases. In this CTE assumption is if there is no previous date (null), or date difference is larger than a year it means screening ended and we treat it as another movie. It's a hypothesis and we'll check it with OMDb.
con.sql("""
WITH lag_date AS(
    SELECT id, title, date, LAG(date) OVER (PARTITION BY title ORDER BY date) AS prev_date FROM raw_revenues
),
date_diff AS (
SELECT 
    id, 
    title, 
    date, 
    prev_date, 
    (date)-(prev_date) AS days_between, 
    CASE WHEN prev_date IS NULL THEN 1
        WHEN date_diff('day', prev_date, date) > 365 THEN 1
        ELSE 0
        END AS new_series
FROM lag_date),
series_number AS (
SELECT
    *,
    SUM(new_series) OVER (PARTITION BY title ORDER BY date) AS series_number
FROM date_diff
)
SELECT title, series_number, MIN(date), MAX(date), year(MIN(date)), COUNT(*) AS count FROM series_number GROUP BY title, series_number  ORDER BY title, series_number


""")

┌────────────────────────────────────────────┬───────────────┬────────────┬────────────┬───────────────────┬───────┐
│                   title                    │ series_number │ min(date)  │ max(date)  │ "year"(min(date)) │ count │
│                  varchar                   │    int128     │    date    │    date    │       int64       │ int64 │
├────────────────────────────────────────────┼───────────────┼────────────┼────────────┼───────────────────┼───────┤
│ '71                                        │             1 │ 2015-02-27 │ 2015-05-07 │              2015 │    70 │
│ '83                                        │             1 │ 2021-12-24 │ 2022-01-16 │              2021 │    14 │
│ '85: The Greatest Team in Football History │             1 │ 2018-01-29 │ 2018-02-07 │              2018 │     2 │
│ 10 Cloverfield Lane                        │             1 │ 2016-03-11 │ 2016-06-02 │              2016 │    84 │
│ 10 Years                                   │             1 │ 2

In [24]:
#connecting to database
resp = requests.get("https://www.omdbapi.com/",
                    params={"apikey": API_KEY, "t": "The Kid"})
data = resp.json()
data

{'Title': 'The Kid',
 'Year': '1921',
 'Rated': 'Passed',
 'Released': '06 Feb 1921',
 'Runtime': '68 min',
 'Genre': 'Comedy, Drama, Family',
 'Director': 'Charles Chaplin',
 'Writer': 'Charles Chaplin',
 'Actors': 'Charles Chaplin, Edna Purviance, Jackie Coogan',
 'Plot': 'The Tramp cares for an abandoned child, but events put their relationship in jeopardy.',
 'Language': 'None, English',
 'Country': 'United States',
 'Awards': '2 wins total',
 'Poster': 'https://m.media-amazon.com/images/M/MV5BZGVmZGM0YTUtOTA1ZC00ZDE4LWE4ZDUtMjAxOTk5YjkwN2Q2XkEyXkFqcGc@._V1_SX300.jpg',
 'Ratings': [{'Source': 'Internet Movie Database', 'Value': '8.2/10'},
  {'Source': 'Rotten Tomatoes', 'Value': '100%'}],
 'Metascore': 'N/A',
 'imdbRating': '8.2',
 'imdbVotes': '144,275',
 'imdbID': 'tt0012349',
 'Type': 'movie',
 'DVD': 'N/A',
 'BoxOffice': 'N/A',
 'Production': 'N/A',
 'Website': 'N/A',
 'Response': 'True'}

In [25]:
#year is important for search (many same titles)
#all information is in type string, so there's a need of casting appropriate type in some fields.

## PIPELINE ##

In [26]:
#Some titles are labeled with year and "Re-release" or "Anniversary" and there would be a problem in next stage
# I need to find as many as I can first

In [27]:
con.sql("""
    SELECT
    title,
    trim(regexp_replace(title,
        '(\d{4}\s*((3D/IMAX|IMAX|3D) Release|[Rr]e-release|release)(\s*\(.*\))?|\d+(st|nd|rd|th) Anniversary.*|Re-release|Recut Release|Limited Domestic Release)$',
        '')) AS base_title
    FROM raw_revenues
    WHERE title ILIKE '%release%'
""")

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
/var/folders/7w/89t_pkpx3lz16gymgp29ygnm0000gn/T/ipykernel_63469/1742333209.py:1: SyntaxWarning: invalid escape sequence '\d'
  con.sql("""


┌─────────────────────────────────────────────────────────────┬──────────────────────────────────────────────┐
│                            title                            │                  base_title                  │
│                           varchar                           │                   varchar                    │
├─────────────────────────────────────────────────────────────┼──────────────────────────────────────────────┤
│ The Polar Express2005 IMAX Release                          │ The Polar Express                            │
│ Day Watch2007 Re-release                                    │ Day Watch                                    │
│ Blade RunnerThe Final Cut (2007 re-release)                 │ Blade RunnerThe Final Cut (2007 re-release)  │
│ Vertigo2018 Re-release                                      │ Vertigo                                      │
│ Goldstone2018 Re-release                                    │ Goldstone                                    │
│

In [28]:
con.sql("""CREATE OR REPLACE TABLE stg_revenue AS
WITH lag_date AS(
    SELECT 
        id, 
        title,
        revenue,
        theaters,
        distributor, 
        date, 
        LAG(date) OVER (PARTITION BY title ORDER BY date) AS prev_date 
    FROM raw_revenues
),
date_diff AS (
    SELECT 
        id, 
        title,
        revenue,
        theaters,
        distributor, 
        date, 
        prev_date, 
        (date)-(prev_date) AS days_between, 
        CASE WHEN prev_date IS NULL THEN 1
            WHEN date_diff('day', prev_date, date) > 365 THEN 1
            ELSE 0
        END AS new_series
FROM lag_date),
series_number AS (
    SELECT
        *,
        SUM(new_series) OVER (PARTITION BY title ORDER BY date) AS series_number
    FROM date_diff
)
    SELECT
        id,
        date, 
        title,
        trim(regexp_replace(title,
                '(\d{4}\s*((3D/IMAX|IMAX|3D) Release|[Rr]e-release|release)(\s*\(.*\))?|\d+(st|nd|rd|th) Anniversary.*|Re-release|Recut Release|Limited Domestic Release)$',
                '')) AS base_title,
        revenue,
        theaters,
        series_number,
        CASE WHEN distributor = '-' THEN NULL
            ELSE distributor
            END AS distributor
    FROM series_number 
    

""")

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
/var/folders/7w/89t_pkpx3lz16gymgp29ygnm0000gn/T/ipykernel_63469/3515567300.py:1: SyntaxWarning: invalid escape sequence '\d'
  con.sql("""CREATE OR REPLACE TABLE stg_revenue AS


In [29]:
con.sql("SELECT COUNT(*) FROM stg_revenue")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       337818 │
└──────────────┘

In [30]:
con.sql("SELECT * FROM stg_revenue LIMIT 5")

┌──────────────────────────────────────┬────────────┬───────────┬────────────┬─────────┬──────────┬───────────────┬──────────────┐
│                  id                  │    date    │   title   │ base_title │ revenue │ theaters │ series_number │ distributor  │
│               varchar                │    date    │  varchar  │  varchar   │  int64  │  int64   │    int128     │   varchar    │
├──────────────────────────────────────┼────────────┼───────────┼────────────┼─────────┼──────────┼───────────────┼──────────────┤
│ c3402ba6-a346-b2ba-53e9-8fdd9012e6e8 │ 2018-01-19 │ 12 Strong │ 12 Strong  │ 5672828 │     3002 │             1 │ Warner Bros. │
│ 21d40b02-e07b-2fb0-f4ba-639094e91791 │ 2018-01-20 │ 12 Strong │ 12 Strong  │ 6838422 │     3002 │             1 │ Warner Bros. │
│ f6091167-593b-5e3d-dc6e-b2c83ccdcda1 │ 2018-01-21 │ 12 Strong │ 12 Strong  │ 3303775 │     3002 │             1 │ Warner Bros. │
│ d17f1c2a-b0e1-fc02-401b-17a40bc93649 │ 2018-01-22 │ 12 Strong │ 12 Strong  │ 1304

In [31]:
con.sql("""
    SELECT title 
    FROM stg_revenue 
    WHERE title ILIKE '%release%'
        OR title ILIKE '%anniversary%'
    GROUP BY title
    """)

┌──────────────────────────────────────────────────────┐
│                        title                         │
│                       varchar                        │
├──────────────────────────────────────────────────────┤
│ The Lobster2016 Re-release                           │
│ The New World2005 Re-release                         │
│ The Polar Express2005 IMAX Release                   │
│ The Santa Clause2020 Re-release                      │
│ Titanic2017 Re-release                               │
│ The Ladykillers2021 Re-release                       │
│ The Tale of The Princess Kaguya2019 Re-release       │
│ The Topp Twins: Untouchable Girls2011 Re-release     │
│ Winged Migration2003 Re-release                      │
│ 2001: A Space Odyssey2018 Re-release                 │
│          ·                                           │
│          ·                                           │
│          ·                                           │
│ Alien2020 Re-release         

In [32]:
#there's 341 movies with "release" or "anniversary" in their name
#labels are "year+ re-relase" or "release" at the end, or 

In [33]:
con.sql("""CREATE OR REPLACE TABLE stg_series AS
    SELECT 
        title, 
        base_title, 
        series_number,
        year(min(date)) AS release_year,
        MIN(date) AS first_date,
        MAX(date) AS last_date,
        CASE WHEN title = base_title THEN 0
            ELSE 1
            END AS is_rerelease,
        SUM(revenue) AS sum_revenue,
        COUNT(*) AS days_in_series
    FROM stg_revenue
    GROUP BY title, base_title, series_number


""")

In [34]:
con.sql("SELECT COUNT(*) FROM stg_series")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         6634 │
└──────────────┘

In [35]:
con.sql("""
    SELECT * FROM stg_revenue WHERE distributor = '-'

""")

┌─────────┬──────┬─────────┬────────────┬─────────┬──────────┬───────────────┬─────────────┐
│   id    │ date │  title  │ base_title │ revenue │ theaters │ series_number │ distributor │
│ varchar │ date │ varchar │  varchar   │  int64  │  int64   │    int128     │   varchar   │
└─────────┴──────┴─────────┴────────────┴─────────┴──────────┴───────────────┴─────────────┘
                                           0 rows                                         

In [36]:
con.sql("""
    SELECT COUNT(*) FROM stg_series

""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         6634 │
└──────────────┘

In [37]:
con.sql("""
    SELECT * FROM stg_series WHERE title = 'The Kid'

""")

┌─────────┬────────────┬───────────────┬──────────────┬────────────┬────────────┬──────────────┬─────────────┬────────────────┐
│  title  │ base_title │ series_number │ release_year │ first_date │ last_date  │ is_rerelease │ sum_revenue │ days_in_series │
│ varchar │  varchar   │    int128     │    int64     │    date    │    date    │    int32     │   int128    │     int64      │
├─────────┼────────────┼───────────────┼──────────────┼────────────┼────────────┼──────────────┼─────────────┼────────────────┤
│ The Kid │ The Kid    │             2 │         2019 │ 2019-03-08 │ 2019-04-18 │            0 │     1508593 │             42 │
│ The Kid │ The Kid    │             1 │         2000 │ 2000-07-07 │ 2000-08-31 │            0 │    63550582 │             50 │
└─────────┴────────────┴───────────────┴──────────────┴────────────┴────────────┴──────────────┴─────────────┴────────────────┘

In [38]:
con.sql("""
    SELECT SUM(revenue) FROM stg_revenue

""")

┌──────────────┐
│ sum(revenue) │
│    int128    │
├──────────────┤
│ 205110995141 │
└──────────────┘

In [39]:
con.sql("""
    SELECT SUM(sum_revenue) FROM stg_series

""")

┌──────────────────┐
│ sum(sum_revenue) │
│      int128      │
├──────────────────┤
│     205110995141 │
└──────────────────┘

In [40]:
#sum of revenues should be the same in both tables (stg_revenue and stg_series)

## OMDb enrichment ##

In [41]:
#using type "movie" to avoid other types of films
#connection, parameters, use year if it's available
def query_omdb(title, year=None):
    params = {"apikey": API_KEY, "t": title, "type": "movie"}
    if year is not None:
        params["y"] = year
    resp = requests.get("https://www.omdbapi.com/", params=params)
    return resp.json()

In [42]:
#check if it's re-release, if not using title and year, if year not working then year-1 (premiere dates, festivals, mistakes in dating)
def find_movie(base_title, release_year, is_rerelease):
    if is_rerelease:
        data = query_omdb(base_title)                        
    else:
        data = query_omdb(base_title, release_year)          
        if data.get("Response") != "True":
            data = query_omdb(base_title, release_year - 1) 

    if data.get("Response") == "True":
        return data
    return None

In [43]:
#testing
data = find_movie("The Kid", 2000, 0)
print(data)

{'Title': 'The Kid', 'Year': '2000', 'Rated': 'PG', 'Released': '07 Jul 2000', 'Runtime': '104 min', 'Genre': 'Comedy, Family, Fantasy', 'Director': 'Jon Turteltaub', 'Writer': 'Audrey Wells', 'Actors': 'Bruce Willis, Spencer Breslin, Emily Mortimer', 'Plot': 'An unhappy, disliked image consultant gets a second shot at life when he is mysteriously confronted by an eight-year-old version of himself.', 'Language': 'English', 'Country': 'United States', 'Awards': '1 win & 3 nominations total', 'Poster': 'https://m.media-amazon.com/images/M/MV5BY2NkNTZhMDItNGJmYy00MmNkLTk1N2MtMTQ3MDFjYTcwZGI4XkEyXkFqcGc@._V1_QL75_UX380_CR0,1,380,562_.jpg', 'Ratings': [{'Source': 'Internet Movie Database', 'Value': '6.1/10'}, {'Source': 'Rotten Tomatoes', 'Value': '49%'}, {'Source': 'Metacritic', 'Value': '45/100'}], 'Metascore': '45', 'imdbRating': '6.1', 'imdbVotes': '46,681', 'imdbID': 'tt0219854', 'Type': 'movie', 'DVD': 'N/A', 'BoxOffice': '$69,691,949', 'Production': 'N/A', 'Website': 'N/A', 'Response

In [44]:
data = find_movie("Nie ma filmu", 2000, 0)
print(data)

None


In [45]:
data = find_movie("Total Recall", 2012, 1)
print(data)

{'Title': 'Total Recall', 'Year': '1990', 'Rated': 'R', 'Released': '01 Jun 1990', 'Runtime': '113 min', 'Genre': 'Action, Adventure, Sci-Fi', 'Director': 'Paul Verhoeven', 'Writer': "Philip K. Dick, Ronald Shusett, Dan O'Bannon", 'Actors': 'Arnold Schwarzenegger, Sharon Stone, Michael Ironside', 'Plot': 'When a man goes in to have virtual vacation memories of the planet Mars implanted in his mind, an unexpected and harrowing series of events forces him to go to the planet for real - or is he?', 'Language': 'English', 'Country': 'United States, Mexico, Netherlands', 'Awards': 'Nominated for 2 Oscars. 7 wins & 16 nominations total', 'Poster': 'https://m.media-amazon.com/images/M/MV5BZjRlMzBkOGMtMTA1Ny00ZWI4LTkzMDItOGVkNWJkN2E4ZTkwXkEyXkFqcGc@._V1_SX300.jpg', 'Ratings': [{'Source': 'Internet Movie Database', 'Value': '7.5/10'}, {'Source': 'Rotten Tomatoes', 'Value': '80%'}, {'Source': 'Metacritic', 'Value': '60/100'}], 'Metascore': '60', 'imdbRating': '7.5', 'imdbVotes': '378,339', 'imdb

In [46]:
#test sample
candidates = con.sql("""
    SELECT title, series_number, base_title, release_year, is_rerelease
    FROM stg_series
    ORDER BY random() 
    LIMIT 50
""").fetchall()

candidates[:3]

[('Black Dynamite', 1, 'Black Dynamite', 2009, 0),
 ('The Lord of the Rings: The Return of the King',
  1,
  'The Lord of the Rings: The Return of the King',
  2003,
  0),
 ('Did You Hear About the Morgans?',
  1,
  'Did You Hear About the Morgans?',
  2009,
  0)]

In [47]:
RAW_PATH = "data/omdb_raw.json"
#if file exists, no API connection

if os.path.exists(RAW_PATH):
    with open(RAW_PATH, encoding="utf-8") as f:
        records = json.load(f)

else:
    candidates = con.sql("""
    SELECT title, series_number, base_title, release_year, is_rerelease
    FROM stg_series
    ORDER BY sum_revenue DESC
    
""").fetchall()
    
    records = []

    for title, series_number, base_title, release_year, is_rerelease in candidates:
        data = find_movie(base_title, release_year, is_rerelease)
        records.append({
            "title": title,
            "series_number": series_number,
            "base_title": base_title,
            "release_year": release_year,
            "response": data,
        })
        print(title)

    with open(RAW_PATH, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False)
    print("***Saved***")

In [48]:
#checking omdb_raw.json file
con.sql("DESCRIBE SELECT * FROM read_json_auto('data/omdb_raw.json')")

┌───────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │                                                                                                                                                                                                                                    column_type                                                                                                                                                                                                                          

In [49]:
#creating table, replacing 'N/A' for nulls
con.sql("""
    CREATE OR REPLACE TABLE stg_omdb AS
        SELECT
            title,
            series_number,
            base_title,
            release_year,
            response.title AS omdb_title,
            TRY_CAST(response.year AS INTEGER) AS omdb_year,
            TRY_CAST(replace(response.runtime, ' min', '') AS INTEGER) AS runtime,
            response.released AS released,
            NULLIF(response.rated, 'N/A') AS rated,
            NULLIF(response.country, 'N/A') AS country,
            response.imdbID AS imdbID,
            TRY_CAST(response.imdbRating AS FLOAT) AS imdb_rating,
            NULLIF(response.genre, 'N/A') AS genre,
            response.type AS type
        FROM read_json_auto('data/omdb_raw.json')
""")

In [50]:
con.sql("SELECT * FROM stg_omdb WHERE type = 'N/A'LIMIT 10")

┌─────────┬───────────────┬────────────┬──────────────┬────────────┬───────────┬─────────┬──────────┬─────────┬─────────┬─────────┬─────────────┬─────────┬─────────┐
│  title  │ series_number │ base_title │ release_year │ omdb_title │ omdb_year │ runtime │ released │  rated  │ country │ imdbID  │ imdb_rating │  genre  │  type   │
│ varchar │     int64     │  varchar   │    int64     │  varchar   │   int32   │  int32  │ varchar  │ varchar │ varchar │ varchar │    float    │ varchar │ varchar │
└─────────┴───────────────┴────────────┴──────────────┴────────────┴───────────┴─────────┴──────────┴─────────┴─────────┴─────────┴─────────────┴─────────┴─────────┘
                                                                               0 rows                                                                              

In [51]:
#how many not matched titles
con.sql("SELECT COUNT(*), COUNT(imdbID) FROM stg_omdb")

┌──────────────┬───────────────┐
│ count_star() │ count(imdbID) │
│    int64     │     int64     │
├──────────────┼───────────────┤
│         6634 │          6185 │
└──────────────┴───────────────┘

In [52]:
#all revenue
con.sql("SELECT SUM(revenue) FROM stg_revenue")

┌──────────────┐
│ sum(revenue) │
│    int128    │
├──────────────┤
│ 205110995141 │
└──────────────┘

In [53]:
#matched imdbid revenue
con.sql("""
    SELECT SUM(s.sum_revenue)
    FROM stg_omdb o
    JOIN stg_series s ON o.title = s.title AND o.series_number = s.series_number
    WHERE o.imdbid IS NOT NULL
""")

┌────────────────────┐
│ sum(s.sum_revenue) │
│       int128       │
├────────────────────┤
│       201872013652 │
└────────────────────┘

In [54]:
#98.4% of revenue is from movies matched with imdb ID
print(201872013652/205110995141)

0.9842086403667759


## dim_movie ##

In [55]:

con.sql(""" 
SELECT 
    imdbID, 
    COUNT(*) 
    FROM stg_omdb GROUP BY imdbID
    HAVING COUNT(*)>1
    ORDER BY COUNT(*) DESC""")


┌────────────┬──────────────┐
│   imdbID   │ count_star() │
│  varchar   │    int64     │
├────────────┼──────────────┤
│ NULL       │          449 │
│ tt0338348  │           12 │
│ tt0107688  │            4 │
│ tt0245429  │            3 │
│ tt5323662  │            3 │
│ tt0347149  │            3 │
│ tt0167261  │            3 │
│ tt0198781  │            3 │
│ tt0087332  │            3 │
│ tt3748528  │            2 │
│     ·      │            · │
│     ·      │            · │
│     ·      │            · │
│ tt11963006 │            2 │
│ tt5226984  │            2 │
│ tt0210945  │            2 │
│ tt0120338  │            2 │
│ tt4587656  │            2 │
│ tt0032138  │            2 │
│ tt0095016  │            2 │
│ tt0116477  │            2 │
│ tt4061196  │            2 │
│ tt0170016  │            2 │
└────────────┴──────────────┘
  96 rows         2 columns
  (20 shown)                

In [56]:
#quick preview
con.sql(""" 
SELECT 
*
    FROM stg_omdb
LIMIT 1""")

┌────────────────────────────────────────────┬───────────────┬────────────────────────────────────────────┬──────────────┬────────────────────────────────────────────┬───────────┬─────────┬─────────────┬─────────┬───────────────────────────────┬───────────┬─────────────┬───────────────────────────┬─────────┐
│                   title                    │ series_number │                 base_title                 │ release_year │                 omdb_title                 │ omdb_year │ runtime │  released   │  rated  │            country            │  imdbID   │ imdb_rating │           genre           │  type   │
│                  varchar                   │     int64     │                  varchar                   │    int64     │                  varchar                   │   int32   │  int32  │   varchar   │ varchar │            varchar            │  varchar  │    float    │          varchar          │ varchar │
├────────────────────────────────────────────┼───────────────┼────────

In [57]:
#movie identity - if possible imdbID, else create title+release_year
con.sql("""
    CREATE OR REPLACE TABLE dim_movie AS
    WITH movies AS (
        SELECT
            COALESCE(imdbID, title || '|' || release_year) AS movie_identity,
            MAX(imdbID)                                    AS imdbID,
            COALESCE(MAX(omdb_title), MIN(title))           AS title,
            COALESCE(MAX(omdb_year), MIN(release_year))     AS release_year,
            MAX(rated)       AS rated,
            MAX(runtime) AS runtime,
            MAX(genre)       AS genre,
            MAX(country)     AS country,
            MAX(imdb_rating) AS imdb_rating
        FROM stg_omdb
        GROUP BY movie_identity)
    SELECT ROW_NUMBER() OVER (ORDER BY movie_identity) AS movie_key, * FROM movies
""")

In [58]:
#checks
con.sql("""
SELECT * FROM dim_movie LIMIT 5""")

┌───────────┬─────────────────────────────────────────────────┬─────────┬────────────────────────────────────────────┬──────────────┬─────────┬─────────┬─────────┬─────────┬─────────────┐
│ movie_key │                 movie_identity                  │ imdbID  │                   title                    │ release_year │  rated  │ runtime │  genre  │ country │ imdb_rating │
│   int64   │                     varchar                     │ varchar │                  varchar                   │    int64     │ varchar │  int32  │ varchar │ varchar │    float    │
├───────────┼─────────────────────────────────────────────────┼─────────┼────────────────────────────────────────────┼──────────────┼─────────┼─────────┼─────────┼─────────┼─────────────┤
│         1 │ '85: The Greatest Team in Football History|2018 │ NULL    │ '85: The Greatest Team in Football History │         2018 │ NULL    │    NULL │ NULL    │ NULL    │        NULL │
│         2 │ 25th Hour|2002                                

In [59]:
#numbers line up
con.sql("""
SELECT COUNT(*), COUNT(movie_identity), COUNT(DISTINCT movie_identity), COUNT(imdbID), COUNT(DISTINCT imdbID) FROM dim_movie""")

┌──────────────┬───────────────────────┬────────────────────────────────┬───────────────┬────────────────────────┐
│ count_star() │ count(movie_identity) │ count(DISTINCT movie_identity) │ count(imdbID) │ count(DISTINCT imdbID) │
│    int64     │         int64         │             int64              │     int64     │         int64          │
├──────────────┼───────────────────────┼────────────────────────────────┼───────────────┼────────────────────────┤
│         6521 │                  6521 │                           6521 │          6072 │                   6072 │
└──────────────┴───────────────────────┴────────────────────────────────┴───────────────┴────────────────────────┘

## dim_date ##

In [60]:
con.sql("""CREATE OR REPLACE TABLE dim_date AS
    WITH days AS (
        SELECT CAST(unnest(generate_series(
            (SELECT MIN(date) FROM stg_revenue),
            (SELECT MAX(date) FROM stg_revenue),
            INTERVAL 1 DAY)) AS DATE) AS date
    )
    SELECT
        date,
        day(date)              AS day_of_month,
        dayname(date)          AS day_of_week,
        month(date)            AS month,
        quarter(date)          AS quarter,
        year(date)             AS year,
        dayofweek(date) IN (0, 6) AS is_weekend
    FROM days""")

In [61]:
con.sql("SELECT COUNT(*) FROM dim_date")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         8466 │
└──────────────┘

In [62]:

con.sql("SELECT COUNT(DISTINCT date) FROM stg_revenue")

┌──────────────────────┐
│ count(DISTINCT date) │
│        int64         │
├──────────────────────┤
│                 8455 │
└──────────────────────┘

In [63]:
#11 dates, have no revenue (no movie)
con.sql("""
        SELECT d.date
        FROM dim_date d
        LEFT JOIN stg_revenue r
        ON d.date = r.date
        WHERE r.revenue IS NULL
""")

┌────────────┐
│    date    │
│    date    │
├────────────┤
│ 2020-03-28 │
│ 2020-03-30 │
│ 2020-03-31 │
│ 2021-05-13 │
│ 2021-09-29 │
│ 2021-10-04 │
│ 2020-04-01 │
│ 2021-10-18 │
│ 2020-03-29 │
│ 2020-04-02 │
│ 2020-12-13 │
└────────────┘
   11 rows  

## fact_revenue ##

In [64]:
con.sql("DESCRIBE stg_revenue")

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ date          │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ title         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ base_title    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ revenue       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ theaters      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ series_number │ HUGEINT     │ YES     │ NULL    │ NULL    │ NULL    │
│ distributor   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [65]:
con.sql("DESCRIBE stg_omdb")

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ title         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ series_number │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ base_title    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ release_year  │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ omdb_title    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ omdb_year     │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ runtime       │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ released      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rated         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ country       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NU

In [66]:
con.sql("""
CREATE OR REPLACE TABLE fact_revenue AS
    SELECT
        r.id,
        m.movie_key,
        r.date,
        r.revenue,   
        r.theaters,  
        r.distributor
    FROM stg_revenue r
    JOIN stg_omdb o 
    ON o.title=r.title AND o.series_number=r.series_number
    JOIN dim_movie m 
    ON m.movie_identity = COALESCE(o.imdbID, o.title || '|' || o.release_year)
""")

In [67]:
con.sql("""
SELECT COUNT(*) FROM stg_revenue
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       337818 │
└──────────────┘

In [68]:
con.sql("""
SELECT COUNT(*) FROM fact_revenue
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       337818 │
└──────────────┘

In [69]:
con.sql("""SELECT COUNT(*), COUNT(id) FROM fact_revenue""")

┌──────────────┬───────────┐
│ count_star() │ count(id) │
│    int64     │   int64   │
├──────────────┼───────────┤
│       337818 │    337818 │
└──────────────┴───────────┘

In [70]:
con.sql("""
SELECT SUM(revenue) FROM stg_revenue
""")

┌──────────────┐
│ sum(revenue) │
│    int128    │
├──────────────┤
│ 205110995141 │
└──────────────┘

In [71]:
con.sql("""
SELECT SUM(revenue) FROM fact_revenue
""")

┌──────────────┐
│ sum(revenue) │
│    int128    │
├──────────────┤
│ 205110995141 │
└──────────────┘

In [72]:
#after checking - same amount of records fact_revenue vs stg_revenue, and the same revenue SUM.

################ dashboard queries ###################

In [73]:
#top 20 grossing movies
con.sql("""
    SELECT 
        m.title, 
        MIN(m.release_year), 
        m.genre, 
        SUM(r.revenue)/1000000 AS revenue_mln
    FROM fact_revenue r
    JOIN dim_movie m ON m.movie_key = r.movie_key
    GROUP BY m.movie_key, m.title, m.genre
    ORDER BY revenue_mln DESC
    LIMIT 20  
        """)

┌───────────────────────────────────────────────┬─────────────────────┬──────────────────────────────┬─────────────┐
│                     title                     │ min(m.release_year) │            genre             │ revenue_mln │
│                    varchar                    │        int64        │           varchar            │   double    │
├───────────────────────────────────────────────┼─────────────────────┼──────────────────────────────┼─────────────┤
│ Star Wars: Episode VII - The Force Awakens    │                2015 │ Action, Adventure, Sci-Fi    │  935.644139 │
│ Avengers: Endgame                             │                2019 │ Action, Adventure, Sci-Fi    │     858.373 │
│ Spider-Man: No Way Home                       │                2021 │ Action, Adventure, Fantasy   │  813.337519 │
│ Avatar                                        │                2009 │ Action, Adventure, Fantasy   │  774.393897 │
│ Top Gun: Maverick                             │               

In [74]:
con.sql("""
    SELECT d.year, SUM(r.revenue) AS revenue
FROM fact_revenue r
JOIN dim_date d ON d.date = r.date
GROUP BY d.year
ORDER BY d.year
        """)

┌───────┬─────────────┐
│ year  │   revenue   │
│ int64 │   int128    │
├───────┼─────────────┤
│  2000 │  5064534151 │
│  2001 │  4495620900 │
│  2002 │  8408060558 │
│  2003 │  8872496660 │
│  2004 │  9019094141 │
│  2005 │  8547690773 │
│  2006 │  8964624969 │
│  2007 │  9450343799 │
│  2008 │  9443637189 │
│  2009 │ 10428615107 │
│    ·  │      ·      │
│    ·  │      ·      │
│    ·  │      ·      │
│  2014 │ 10154417699 │
│  2015 │ 10905366386 │
│  2016 │ 11216540850 │
│  2017 │ 10946049281 │
│  2018 │ 11760330043 │
│  2019 │ 11213866258 │
│  2020 │  1984522602 │
│  2021 │  4067918034 │
│  2022 │  7274024712 │
│  2023 │  1194491623 │
└───────┴─────────────┘
  24 rows   2 columns
  (20 shown)          

In [75]:
#average revenue per movie was highest in 2000, 2001 and 2010
#important: 2000-2001 have only 85 and 110 movies (200+ later), so it's data coverage, not a trend
con.sql("""
    SELECT
            m.release_year,
            SUM(r.revenue) / COUNT(DISTINCT r.movie_key) / 1000000 AS avg_revenue_mln,
            COUNT(DISTINCT r.movie_key)                        AS movies
        FROM fact_revenue r
        JOIN dim_movie m ON m.movie_key = r.movie_key
        WHERE m.release_year BETWEEN 2000 AND 2023
        GROUP BY m.release_year
        ORDER BY m.release_year
    
     
        """)

┌──────────────┬────────────────────┬────────┐
│ release_year │  avg_revenue_mln   │ movies │
│    int64     │       double       │ int64  │
├──────────────┼────────────────────┼────────┤
│         2000 │  60.34464249411765 │     85 │
│         2001 │ 45.342361981818186 │    110 │
│         2002 │   39.5767160921659 │    217 │
│         2003 │  36.90986155319149 │    235 │
│         2004 │  41.64406778703704 │    216 │
│         2005 │  37.85739357079646 │    226 │
│         2006 │  34.57545867567568 │    259 │
│         2007 │  35.13485388518518 │    270 │
│         2008 │  34.53552967272727 │    275 │
│         2009 │  41.52239153696498 │    257 │
│           ·  │          ·         │     ·  │
│           ·  │          ·         │     ·  │
│           ·  │          ·         │     ·  │
│         2014 │  32.26048656521739 │    322 │
│         2015 │ 34.278496647435894 │    312 │
│         2016 │ 27.436837656174337 │    413 │
│         2017 │ 23.565907809935204 │    463 │
│         201

In [76]:
#not even one movie in TOP100 revenue without United States in countries.
con.sql("""
WITH sum_rev AS (
    SELECT movie_key, SUM(revenue) AS revenue_per
    FROM fact_revenue
    GROUP BY movie_key
    ORDER BY revenue_per DESC
    LIMIT 100
),
ranking_rev AS (
    SELECT
        ROW_NUMBER() OVER (ORDER BY sr.revenue_per DESC) AS ranking,
        sr.revenue_per,
        m.country,
        CASE
            WHEN m.country ILIKE '%United States%' THEN 1
            WHEN m.country IS NULL THEN NULL
            ELSE 0
        END AS is_usa
    FROM sum_rev sr
    JOIN dim_movie m ON m.movie_key = sr.movie_key
)
SELECT COUNT(*)
FROM ranking_rev
WHERE is_usa = 0
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│            0 │
└──────────────┘

In [77]:
#if we assume removing 'United States" as first country of the list then there's 18 movies left
con.sql("""
WITH sum_rev AS (
    SELECT movie_key, SUM(revenue) AS revenue_per
    FROM fact_revenue
    GROUP BY movie_key
    ORDER BY revenue_per DESC
    LIMIT 100
),
ranking_rev AS (
    SELECT
        ROW_NUMBER() OVER (ORDER BY sr.revenue_per DESC) AS ranking,
        sr.revenue_per,
        m.country,
        CASE
            WHEN m.country ILIKE 'United States%' THEN 1
            WHEN m.country IS NULL THEN NULL
            ELSE 0
        END AS is_usa
    FROM sum_rev sr
    JOIN dim_movie m ON m.movie_key = sr.movie_key
)
SELECT COUNT(*)
FROM ranking_rev
WHERE is_usa = 0
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│           18 │
└──────────────┘

In [78]:
#best day of the week in mln
con.sql("""SELECT
    d.day_of_week,
    (SUM(r.revenue) / COUNT(DISTINCT d.date))/1000000 AS avg_revenue_per_day_mln
FROM fact_revenue r
JOIN dim_date d ON d.date = r.date
GROUP BY d.day_of_week
ORDER BY avg_revenue_per_day_mln DESC""")

┌─────────────┬─────────────────────────┐
│ day_of_week │ avg_revenue_per_day_mln │
│   varchar   │         double          │
├─────────────┼─────────────────────────┤
│ Saturday    │       47.47905624731183 │
│ Friday      │        38.8152653101737 │
│ Sunday      │       32.54674318377483 │
│ Monday      │      13.769183111847555 │
│ Tuesday     │      13.234690052980133 │
│ Wednesday   │      12.159182592377796 │
│ Thursday    │      11.749504244407623 │
└─────────────┴─────────────────────────┘

In [79]:
#closing duckDB connection
con.close()

Pipeline finished. To open the dashboard, run in the terminal: streamlit run app.py